In [12]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

import pennylane as qml

from tqdm import tqdm

from medmnist import TissueMNIST
from medmnist import INFO

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Using: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("CUDA is not available. Using CPU.")

Using: NVIDIA GeForce RTX 4070 Ti SUPER
CUDA: 12.4


In [14]:
info = INFO['tissuemnist']
data_flag = 'tissuemnist'
DataClass = TissueMNIST

task = info['task']  
n_channels = info['n_channels']
n_classes = len(info['label'])

print(f"Number of classes:", n_classes)
print(f"Number of channels: {n_channels}")

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),  
    transforms.RandomRotation(degrees=15),  
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

Number of classes: 8
Number of channels: 1


In [15]:
data_train = DataClass(split='train', transform=train_transform, download=True, size=28)
data_test = DataClass(split='test', transform=eval_transform, download=True, size=28)
data_eval = DataClass(split='val', transform=eval_transform, download=True, size=28)

Using downloaded and verified file: /home/eflammere/.medmnist/tissuemnist.npz
Using downloaded and verified file: /home/eflammere/.medmnist/tissuemnist.npz
Using downloaded and verified file: /home/eflammere/.medmnist/tissuemnist.npz


## **Dataloader: Train, Test and Validation**

The following code snippet demonstrates how to load the training, test, and validation splits of the dataset:


In [16]:
batch_size = 128

dataloader_train = DataLoader(dataset=data_train, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(dataset=data_test, batch_size=batch_size, shuffle=False)
dataloader_eval = DataLoader(dataset=data_eval, batch_size=batch_size, shuffle=False)

print(f"\nNumber of images in training dataset: {len(data_train)}")
print(f"Number of images in test dataset: {len(data_test)}")
print(f"Number of images in validation dataset: {len(data_eval)}")



Number of images in training dataset: 165466
Number of images in test dataset: 47280
Number of images in validation dataset: 23640


# **Quanvolution**

Designed to process an image by applying a quantum circuit to extract features from it. It works similarly to a convolutional layer in a neural network, but instead of using traditional mathematical filters, it leverages a quantum circuit to process small patches of the image and generate new features.

In [17]:
def quanvolution(image, circuit, patch_size, n_qubits):
    if image.ndim == 2:
        image = np.expand_dims(image, axis=-1)
    
    height_patches = image.shape[0] // patch_size
    width_patches = image.shape[1] // patch_size
    
    out = np.zeros((height_patches, width_patches, n_qubits))
    
    for j in range(height_patches):
        for k in range(width_patches):
            patch = []
            for i in range(patch_size):
                for l in range(patch_size):
                    if (j * patch_size + i < image.shape[0]) and (k * patch_size + l < image.shape[1]):
                        patch.append(image[j * patch_size + i, k * patch_size + l, 0])
                    else:
                        patch.append(0)
            
            q_results = circuit(patch)
            
            for c in range(n_qubits):
                out[j, k, c] = q_results[c]
    
    return out

def quanvolution_batch(images, circuit, patch_size, n_qubits):
    batch_size = images.shape[0]
    processed = [
        quanvolution(images[i].detach().cpu().numpy(), circuit, patch_size, n_qubits)
        for i in range(batch_size)
    ]

    processed = np.array(processed)
    return torch.tensor(processed, dtype=torch.float32).to(images.device)

In [18]:
n_qubits = 4
n_layers = 1

rand_params = np.random.uniform(high=2 * np.pi, size=(n_layers, n_qubits))

def get_device(n_qubits):
    return qml.device("lightning.gpu", wires=n_qubits)

def define_circuit(rand_params):
    dev = get_device(n_qubits)

    @qml.qnode(dev, interface='torch')
    def circuit(phi):
        for j in range(n_qubits):
            qml.RY(np.pi * phi[j], wires=j)

        qml.templates.layers.RandomLayers(rand_params, list(range(n_qubits)))

        return [qml.expval(qml.PauliZ(j)) for j in range(n_qubits)]

    return circuit

rand_circuit = define_circuit(rand_params)

phi = np.random.uniform(size=n_qubits)

result = rand_circuit(phi)

expanded_circuit = rand_circuit.qtape.expand()
print(expanded_circuit.draw())

0: ──RY────────────────┤  <Z>
1: ──RY──RY─╭●─────────┤  <Z>
2: ──RY──RX─│───RZ──RX─┤  <Z>
3: ──RY─────╰X─────────┤  <Z>


In [19]:
class Quanvolution(nn.Module):
    def __init__(self, rand_params, output_size = (14, 14), patch_size = 2, n_qubits = 4, num_classes = 8):
        super(Quanvolution, self).__init__()
        self.output_size = output_size
        self.patch_size = patch_size
        self.n_qubits = n_qubits
        self.num_classes = num_classes
        
        self.circuit = define_circuit(rand_params)

        self.flatten = nn.Flatten()
        self.fc = nn.Linear(output_size[0] * output_size[1] * n_qubits, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = quanvolution_batch(x, self.circuit, self.patch_size, self.n_qubits)
        x = torch.relu(x)
        x = self.flatten(x)
        x = self.fc(x)
        return F.log_softmax(x, dim=1)

In [20]:
model = Quanvolution(rand_params).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
criterion = nn.CrossEntropyLoss().to(device)
epochs = 20

In [ ]:
best_model_path = "/home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/notebooks/quantum/best_model.pth"

best_val_acc = 0.0 

train_losses = []
val_losses = []
val_accuracies = []
val_precisions = []
val_recalls = []
val_f1_scores = []
val_aucs = []

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")

    model.train()
    total_loss = 0.0
    print("\n[Training]")
    for batch_idx, (images, labels) in enumerate(tqdm(dataloader_train, desc="Training Batches", bar_format="{desc}: {n}/{total}")):
        images, labels = images.squeeze(1).to(device), labels.squeeze().to(device)

        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        batch_accuracy = accuracy_score(labels.cpu().numpy(), output.argmax(dim=1).cpu().numpy())
        print(f"Loss: {loss.item():.4f}, Accuracy: {batch_accuracy:.3f}")

    epoch_train_loss = total_loss / len(dataloader_train)
    train_losses.append(epoch_train_loss)
    print(f"Epoch {epoch + 1} Training Loss: {epoch_train_loss:.4f}")

    scheduler.step()

    model.eval()
    val_loss = 0.0
    val_labels, val_predictions = [], []

    print("\n[Validation]")
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(tqdm(dataloader_eval, desc="Validation Batches", bar_format="{desc}: {n}/{total}")):
            images, labels = images.squeeze(1).to(device), labels.squeeze().to(device)
            output = model(images)
            loss = criterion(output, labels)
            val_loss += loss.item()

            val_labels.append(labels)
            val_predictions.append(output)

            batch_accuracy = accuracy_score(labels.cpu().numpy(), output.argmax(dim=1).cpu().numpy())
            print(f"Loss: {loss.item():.4f}, Accuracy: {batch_accuracy:.3f}")

    epoch_val_loss = val_loss / len(dataloader_eval)
    val_losses.append(epoch_val_loss)
    val_labels = torch.cat(val_labels)
    val_predictions = torch.cat(val_predictions)

    val_labels = val_labels.cpu().numpy()
    val_preds = val_predictions.argmax(dim=1).cpu().numpy()
    val_probs = torch.softmax(val_predictions, dim=1).cpu().numpy()

    val_accuracy = accuracy_score(val_labels, val_preds)
    val_precision = precision_score(val_labels, val_preds, average="macro", zero_division=0)
    val_recall = recall_score(val_labels, val_preds, average="macro", zero_division=0)
    val_f1 = f1_score(val_labels, val_preds, average="macro", zero_division=0)

    try:
        val_auc = roc_auc_score(val_labels, val_probs, multi_class="ovr")
    except ValueError:
        val_auc = float("nan")

    val_accuracies.append(val_accuracy)
    val_precisions.append(val_precision)
    val_recalls.append(val_recall)
    val_f1_scores.append(val_f1)
    val_aucs.append(val_auc)

    print(
        f"\nEpoch {epoch + 1} Summary:\n"
        f"Train Loss: {epoch_train_loss:.4f}, "
        f"Val Loss: {epoch_val_loss:.4f}, "
        f"Accuracy: {val_accuracy:.3f}, "
        f"Precision: {val_precision:.3f}, "
        f"Recall: {val_recall:.3f}, "
        f"F1: {val_f1:.3f}, "
        f"AUC: {val_auc:.3f}"
    )

    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        torch.save(model.state_dict(), best_model_path)
        print(f"New best model saved with validation accuracy of {best_val_acc*100:.4f})!")

print(f"Best model saved at {best_model_path} with validation accuracy of {best_val_acc*100:.4f}")



Epoch 1/20

[Training]


Training Batches: 1/1293

Loss: 2.1194, Accuracy: 0.117


Training Batches: 2/1293

Loss: 1.7819, Accuracy: 0.383


...

Loss: 2.4218, Accuracy: 0.297


Training Batches: 4/1293

Loss: 2.1007, Accuracy: 0.344


Training Batches: 5/1293

Loss: 1.7926, Accuracy: 0.281


In [ ]:
model_path = "/home/eflammere/BreastCancerQuanvolution/Quantum/checkpoints/BreastMNIST/28x28/best_model.pth"
model.load_state_dict(torch.load(model_path, weights_only=True))

model.eval()

test_loss = 0.0
test_labels, test_predictions = [], []

with torch.no_grad():
    for images, labels in dataloader_test:
        images, labels = images.squeeze(1).to(device), labels.squeeze().to(device)
        output = model(images)
        loss = criterion(output, labels)
        test_loss += loss.item()
        
        test_labels.append(labels)
        test_predictions.append(output)

test_labels = torch.cat(test_labels)
test_predictions = torch.cat(test_predictions)

test_probs = torch.softmax(test_predictions, dim=1).cpu().numpy()

test_labels_np = test_labels.cpu().numpy()
test_preds_np = test_predictions.argmax(dim=1).cpu().numpy()

test_accuracy = accuracy_score(test_labels_np, test_preds_np)
test_precision = precision_score(test_labels_np, test_preds_np, average="macro", zero_division=0)
test_recall = recall_score(test_labels_np, test_preds_np, average="macro", zero_division=0)
test_f1 = f1_score(test_labels_np, test_preds_np, average="macro", zero_division=0)

try:
    test_auc = roc_auc_score(test_labels_np, test_probs, multi_class="ovr")
except ValueError:
    test_auc = float("nan")

print("\nFinal Test Evaluation:")
print(f"Test Loss: {test_loss / len(dataloader_test):.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"Test AUC: {test_auc:.4f}")

NameError: name 'model' is not defined